In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from threading import Lock
import requests

In [ ]:
import pyarrow as pa

conn = notebookutils.data.connect_to_artifact("WH_HDX_DE_Silver", "<dev-workspace-guid>", "Warehouse")
sql = """
SELECT TOP (100) [Process],
			[Status],
			[LastRunTime],
			[LastSuccess],
			[RunInstanceId],
			[LastCheckedTime],
			[RetryCount]
FROM [WH_HDX_DE_Silver].[orch].[processexecutionstatus]
"""
cursor = conn.execute(sql)
columns = [column[0] for column in cursor.description]
data = cursor.fetchall()
columnar_data = list(zip(*data))
if len(columnar_data) > 0:
	arrow_table = pa.Table.from_arrays([pa.array(col) for col in columnar_data], columns)
	display(arrow_table.to_pandas())
else:
	print('empty table')


In [ ]:
import pyarrow as pa

conn = notebookutils.data.connect_to_artifact("WH_HDX_DE_Silver", "<dev-workspace-guid>", "Warehouse")
sql = """
SELECT TOP (100) [timestamp],
			[process],
			[status],
			[message],
			[run_id]
FROM [WH_HDX_DE_Silver].[orch].[orchestrator_run_logs]
"""
cursor = conn.execute(sql)
columns = [column[0] for column in cursor.description]
data = cursor.fetchall()
columnar_data = list(zip(*data))
if len(columnar_data) > 0:
	arrow_table = pa.Table.from_arrays([pa.array(col) for col in columnar_data], columns)
	display(arrow_table.to_pandas())
else:
	print('empty table')


- Gets the environment from the current workspace name to inform where to run all processes.
- Resolves the orchestrator warehouse in the *current* workspace.
- Sets up the shared SQL connection and the lock that serializes it.
- Sets the limit for retries.

In [ ]:
nb_context = notebookutils.runtime.context
current_workspace_id = nb_context['currentWorkspaceId']
current_workspace_name = nb_context.get("currentWorkspaceName").lower()

if "dev" in current_workspace_name:
    current_env = 'DEV'
elif "qa" in current_workspace_name:
    current_env = 'QA'
elif "uat" in current_workspace_name:
    current_env = 'UAT'
elif "prd" in current_workspace_name:
    current_env = 'PRD'
else:
    raise ValueError(f"Unknown workspace environment: {current_workspace_name}")

RETRY_LIMIT = 3

# Timezone that dependencygraph.RunTimes cron expressions are evaluated in.
# UTC keeps `now` in the same frame as LastSuccess (derived from the pipeline
# API's startTimeUtc) and the warehouse's CURRENT_TIMESTAMP, so the eligibility
# comparison is apples-to-apples. Switching this to ZoneInfo("America/Chicago")
# would move every firing time - treat that as a schedule change, not a tidy-up.
SCHEDULE_TIMEZONE = timezone.utc

# The orchestrator warehouse always lives in the workspace this notebook runs in
# (WS_HDX_DE_DEV / _QA / _UAT / _PRD), so resolve it at runtime. A hardcoded
# workspace id cannot be remapped by deployment pipelines or git deployment
# rules, and would point QA/UAT/PRD runs at the DEV orchestrator tables while
# still triggering pipelines in their own environment.
silver_wh_conn = notebookutils.data.connect_to_artifact(
    "WH_HDX_DE_Silver",
    current_workspace_id,
    "Warehouse"
)

# One ODBC connection is shared by every thread in the pools below. A single
# connection cannot service concurrent cursors, and commit() is connection-wide
# (it would commit another thread's partially written statement), so all
# statements funnel through this lock.
sql_lock = Lock()


def execute_sql(sql, params=None):
    """Run a write statement. Returns rowcount, read *before* the commit."""
    with sql_lock:
        if params is None:
            cursor = silver_wh_conn.execute(sql)
        else:
            cursor = silver_wh_conn.execute(sql, tuple(params))
        try:
            rowcount = cursor.rowcount
            silver_wh_conn.commit()
        finally:
            # Close inside the lock. conn.execute() opens a NEW cursor every
            # call, and without MARS this connection allows only one active
            # statement - a cursor still open when the next thread takes the
            # lock fails with "Connection is busy with results for another
            # command". Releasing the lock is not enough; the cursor has to be
            # gone too, and relying on frame teardown to do it is a race.
            cursor.close()
        return rowcount


def fetch_as_dicts(sql, params=None):
    """Run a query, return all rows as dicts. Lock is held across the fetch."""
    with sql_lock:
        if params is None:
            cursor = silver_wh_conn.execute(sql)
        else:
            cursor = silver_wh_conn.execute(sql, tuple(params))
        try:
            columns = [c[0] for c in cursor.description]
            rows = cursor.fetchall()
        finally:
            # fetchall() has already materialised the rows, so the cursor must
            # be closed here rather than left to the frame: the dict building
            # below runs outside the lock and would otherwise keep the
            # connection busy while another thread is executing on it.
            cursor.close()

    return [dict(zip(columns, row)) for row in rows]

Acquires the orchestrator lock. Stamps the lock row with a value unique to this
run and reads it back - if our stamp is what landed, we own the lock. This does
not depend on MERGE's rowcount, which is driver-dependent and was previously
read after commit (where it can report -1 and fake an acquisition).

In [ ]:
LOCK_TIMEOUT_MINUTES = 30

# datetime2(6) on AcquiredAt matches Python's microsecond precision, so this
# round-trips exactly.
lock_stamp = datetime.utcnow()

execute_sql(
    f"""
    MERGE orch.orchestrator_lock AS target
    USING (SELECT 'orchestrator' AS LockName) AS source
    ON target.LockName = source.LockName

    WHEN MATCHED
        AND target.AcquiredAt < DATEADD(MINUTE, -{LOCK_TIMEOUT_MINUTES}, CURRENT_TIMESTAMP)
    THEN
        UPDATE SET AcquiredAt = CAST(? AS DATETIME2)

    WHEN NOT MATCHED THEN
        INSERT (LockName, AcquiredAt)
        VALUES (source.LockName, CAST(? AS DATETIME2));
    """,
    (lock_stamp, lock_stamp)
)

held = fetch_as_dicts("""
    SELECT AcquiredAt
    FROM orch.orchestrator_lock
    WHERE LockName = 'orchestrator'
""")

if not held or held[0]["AcquiredAt"] != lock_stamp:
    held_str = str(held[0]["AcquiredAt"]) if held else "unknown"
    print(
        f"Orchestrator lock held since {held_str} - "
        "another run is still active. Exiting."
    )
    notebookutils.notebook.exit("skipped: lock held")

print(f"Lock acquired at {lock_stamp.isoformat()}")

Function to add logs to the orch.orchestrator_run_logs table. Run ID is an
optional field. A NULL message or run_id binds as SQL NULL.

In [ ]:
def log_event(process, status, message, run_id=None):
    execute_sql(
        """
        INSERT INTO orch.orchestrator_run_logs
        (
            [timestamp],
            process,
            status,
            message,
            run_id
        )
        VALUES (?, ?, ?, ?, ?);
        """,
        (
            datetime.utcnow().isoformat(),
            process,
            status,
            message,
            run_id
        )
    )

Checks the dependency graph for circular dependencies. If found, the row is removed from dependency_rows.

In [ ]:
def break_cycles(dependency_rows):
    graph = defaultdict(list)
    edges = set()

    for row in dependency_rows:

        dependency = row.get("Dependency")
        process = row.get("Process")

        if dependency:
            graph[dependency].append(process)
            edges.add((dependency, process))

    visited = set()
    removed_edges = []

    def visit(node, path):

        if node in path:

            cycle_start = path.index(node)
            cycle_path = path[cycle_start:] + [node]

            u, v = cycle_path[-2], cycle_path[-1]

            if (u, v) in edges:
                removed_edges.append((u, v))
                graph[u].remove(v)
                edges.remove((u, v))

            return

        if node in visited:
            return

        path.append(node)

        for neighbor in list(graph[node]):
            visit(neighbor, path)

        path.pop()
        visited.add(node)

    for node in list(graph.keys()):
        visit(node, [])

    removed_edge_set = set(removed_edges)

    cleaned_rows = [
        row
        for row in dependency_rows
        if (
            row.get("Dependency"),
            row.get("Process")
        ) not in removed_edge_set
    ]

    return cleaned_rows, removed_edges

Checks the retry count for eligible processes. If greater than or equal to the limit, the process is marked as failed until manual remediation and reset of the retry count to NULL.

In [ ]:
def apply_retry_limits(eligible_processes, retry_count_map):
    retry_blocked = [
        proc
        for proc in eligible_processes
        if retry_count_map.get(proc, 0) >= RETRY_LIMIT
    ]

    for proc in retry_blocked:
        execute_sql(
            """
            UPDATE orch.processexecutionstatus
            SET Status = 'Failed'
            WHERE Process = ?
            """,
            (proc,)
        )

    return retry_blocked

Uses the workspace name to query to the API and get the workspace ID.
Cached, and follows continuationUri so a workspace past the first page is found.

In [ ]:
workspace_cache = {}


def get_workspace_id_by_name(workspace_name: str) -> str:

    if workspace_name in workspace_cache:
        return workspace_cache[workspace_name]

    workspace_url = "https://api.fabric.microsoft.com/v1/workspaces"

    headers = {
        "Authorization": f"Bearer {notebookutils.credentials.getToken('https://api.fabric.microsoft.com')}",
        "Content-Type": "application/json"
    }

    while workspace_url:

        response = requests.get(workspace_url, headers=headers)
        response.raise_for_status()

        payload = response.json()

        workspace = next(
            (
                w
                for w in payload.get("value", [])
                if w["displayName"] == workspace_name
            ),
            None
        )

        if workspace:
            workspace_cache[workspace_name] = workspace["id"]
            return workspace["id"]

        workspace_url = payload.get("continuationUri")

    raise ValueError(f"Workspace '{workspace_name}' not found.")

Uses the workspace ID and pipeline name to query the API and get the pipeline ID.
Cached, and follows continuationUri.

In [ ]:
pipeline_cache = {}


def get_pipeline_id(workspace_id: str, pipeline_name: str) -> str:

    cache_key = f"{workspace_id}:{pipeline_name}"

    if cache_key in pipeline_cache:
        return pipeline_cache[cache_key]

    list_url = (
        f"https://api.fabric.microsoft.com/v1/workspaces/"
        f"{workspace_id}/dataPipelines"
    )

    headers = {
        "Authorization": (
            f"Bearer "
            f"{notebookutils.credentials.getToken('https://api.fabric.microsoft.com')}"
        ),
        "Content-Type": "application/json"
    }

    while list_url:

        response = requests.get(list_url, headers=headers)
        response.raise_for_status()

        payload = response.json()

        pipeline = next(
            (
                p
                for p in payload.get("value", [])
                if p["displayName"] == pipeline_name
            ),
            None
        )

        if pipeline:
            pipeline_cache[cache_key] = pipeline["id"]
            return pipeline["id"]

        list_url = payload.get("continuationUri")

    raise ValueError(
        f"Pipeline '{pipeline_name}' not found in workspace '{workspace_id}'."
    )

Resolves which pipeline runs a given process: gold stored procs go through
PL_Run_Gold_Proc, other stored procs through PL_Run_Proc, and everything else
is its own pipeline. Shared by trigger and poll so the two cannot disagree.

In [ ]:
def resolve_pipeline_name(process, is_stored_proc, is_gold):
    if not is_stored_proc:
        return process

    return 'PL_Run_Gold_Proc' if is_gold else 'PL_Run_Proc'

Takes process, workspace_prefix, is_stored_proc, and last_modified. Uses workspace_prefix and current env to create the workspace name. The workspace name is used to get the workspace ID. The workspace ID and process name are used to get the pipeline ID. The IDs are used to construct the URL for the API call to trigger the given pipeline. If the process is a stored proc, a payload is sent with the call to set the pipeline parameters for last_modified and which stored proc to run.

Returns T/F if successfully triggered, as well as the run ID.

In [ ]:
def trigger_pipeline(process, isGold, lookbackRequired, workspace_prefix, is_stored_proc, last_modified):
    workspace_name = workspace_prefix + current_env

    workspace_id = get_workspace_id_by_name(workspace_name)

    if lookbackRequired and last_modified:
        last_modified -= timedelta(hours=3)

    headers = {
        "Authorization": f"Bearer {notebookutils.credentials.getToken('https://api.fabric.microsoft.com')}",
        "Content-Type": "application/json"
    }

    pipeline_id = get_pipeline_id(
        workspace_id,
        resolve_pipeline_name(process, is_stored_proc, isGold)
    )

    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{pipeline_id}/jobs/instances?jobType=Pipeline"

    if is_stored_proc:
        payload = {
            "executionData": {
                "parameters": {
                    "StoredProcName": f"[dbo].[{process}]",
                    "LastModified": f"{last_modified}" if last_modified else "1900-01-01"

                }
            }
        }

    else:
        payload = {}

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 202:
        job_instance_id = response.headers.get("x-ms-job-id")
        print(f"Triggered pipeline '{process}' with Job ID: {job_instance_id}", flush=True)
        return True, job_instance_id
    else:
        print(f"Failed to trigger pipeline '{process}': {response.status_code} - {response.text}", flush=True)
        return False, None

Takes workspace_id, run_id, is_stored_proc, is_gold and process. Uses workspace_id, run_id and process to create URL for and API call to get the status of the given run_id.

Returns the status (Completed, Failed, Canceled, Running), start time (determines the cutoff for last_modified), and the failure message (if Failed).

In [ ]:
def check_pipeline_status(workspace_id, run_id, is_stored_proc, process, is_gold=False):
    # Must resolve the same pipeline trigger_pipeline used, or the job instance
    # is looked up under the wrong pipeline id - previously gold stored procs
    # were triggered on PL_Run_Gold_Proc but polled against PL_Run_Proc.
    pipeline_id = get_pipeline_id(
        workspace_id,
        resolve_pipeline_name(process, is_stored_proc, is_gold)
    )

    status_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{pipeline_id}/jobs/instances/{run_id}"
    headers = {
        "Authorization": f"Bearer {notebookutils.credentials.getToken('https://api.fabric.microsoft.com')}",
        "Content-Type": "application/json"
    }
    response = requests.get(status_url, headers=headers)
    job = response.json()
    job_status = job.get('status')
    start_time = job.get('startTimeUtc')

    failure_message = None
    if job_status == 'Failed':
        failure_reason = job.get('failureReason', {})
        failure_message = failure_reason.get('message')

    return(job_status, start_time, failure_message)

In [ ]:
def get_pipeline_activity_run_details(workspace_id, run_id):

    status_url = (
        f"https://api.fabric.microsoft.com/v1/workspaces/"
        f"{workspace_id}/datapipelines/pipelineruns/"
        f"{run_id}/queryactivityruns"
    )

    headers = {
        "Authorization": (
            f"Bearer "
            f"{notebookutils.credentials.getToken('https://api.fabric.microsoft.com')}"
        ),
        "Content-Type": "application/json"
    }

    body = {
        "lastUpdatedBefore": datetime.now(timezone.utc).isoformat()
    }

    run_info = None

    try:

        response = requests.post(
            status_url,
            headers=headers,
            json=body
        )

        response.raise_for_status()

        data = response.json()

        activities = data.get("value", [])

        if not activities:
            return None

        activity = next(
            (a for a in activities if a is not None),
            None
        )

        if activity is None:
            return None

        output = activity.get("output") or {}

        rows_read = output.get("rowsRead")
        rows_copied = output.get("rowsCopied")
        data_read = output.get("dataRead")
        data_written = output.get("dataWritten")

        if any(
            x is not None
            for x in (
                rows_read,
                rows_copied,
                data_read,
                data_written
            )
        ):
            run_info = (
                f"Rows Read: {rows_read}, "
                f"Rows Copied: {rows_copied}, "
                f"Data Read: {data_read} bytes, "
                f"Data Written: {data_written} bytes"
            )

    except Exception as e:
        print(
            f"Activity run query failed or returned no data: {e}",
            flush=True
        )

    return run_info

1. Checks the status table for running processes.
2. For each running process checks the status of the last run.
3. Updates the status table with new status/timestamps

In [ ]:
def poll_single_process(row, results):

    process = row["Process"]
    run_id = row["RunInstanceId"]
    retry_count = row["RetryCount"] or 0

    if not run_id:
        results.append(
            (
                process,
                "Failed",
                None,
                retry_count + 1,
                "No RunInstanceId — trigger likely failed",
                "",
                ""
            )
        )
        return

    try:

        running_proc_info = dependency_map.get(process)

        if not running_proc_info:
            raise ValueError(
                f"No dependency info found for {process}"
            )

        workspace_name = (
            running_proc_info["WorkspacePrefix"]
            + current_env
        )

        workspace_id = get_workspace_id_by_name(
            workspace_name
        )

        is_stored_proc = running_proc_info["isStoredPro"]
        is_gold = running_proc_info["isGold"]

        run_status, start_time, failure_message = (
            check_pipeline_status(
                workspace_id,
                run_id,
                is_stored_proc,
                process,
                is_gold
            )
        )

        normalized_status = (
            "Completed"
            if run_status == "Completed"
            else "Failed"
            if run_status in ("Failed", "Cancelled")
            else "Running"
        )

        try:
            run_info = get_pipeline_activity_run_details(
                workspace_id,
                run_id
            )

        except Exception as e:
            print(
                f"Activity run details unavailable: {e}",
                flush=True
            )
            run_info = None

        final_start_time = None
        final_run_info = run_info or ""
        new_retry_count = retry_count

        if normalized_status == "Completed":

            if isinstance(start_time, str):

                try:
                    start_time = datetime.fromisoformat(
                        start_time
                    )
                except Exception:
                    start_time = None

            # startTimeUtc parses to a tz-aware datetime on 3.12. LastSuccess is
            # stored and compared as naive UTC, so flatten it into that frame
            # here rather than letting it reach SQL as a "+00:00" string whose
            # offset the DATETIME2 conversion silently drops.
            if start_time is not None and start_time.tzinfo is not None:
                start_time = (
                    start_time
                    .astimezone(timezone.utc)
                    .replace(tzinfo=None)
                )

            final_start_time = start_time
            new_retry_count = 0

        elif normalized_status == "Failed":

            new_retry_count = retry_count + 1

        else:

            final_run_info = ""

        results.append(
            (
                process,
                normalized_status,
                final_start_time,
                new_retry_count,
                failure_message or "",
                run_id or "",
                final_run_info
            )
        )

    except Exception as e:

        print(
            f"Polling error for {process}: {e}",
            flush=True
        )

        results.append(
            (
                process,
                "Failed",
                None,
                retry_count + 1,
                str(e),
                run_id or "",
                ""
            )
        )

In [ ]:
def poll_running_processes():
    running_processes = fetch_as_dicts("""
        SELECT
            Process,
            RunInstanceId,
            RetryCount
        FROM orch.processexecutionstatus
        WHERE Status = 'Running'
    """)

    results = []

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [
            executor.submit(poll_single_process, row, results)
            for row in running_processes
        ]

        for future in as_completed(futures):
            future.result()

    if not results:
        return

    # Explicit CASTs: inside a VALUES table constructor a bare ? has no type
    # context, and a NULL LastSuccess would leave the source column's type
    # ambiguous.
    row_clause = (
        "(CAST(? AS VARCHAR(4000)), CAST(? AS VARCHAR(4000)), "
        "CAST(? AS DATETIME2), CAST(? AS INT))"
    )

    row_clauses = []
    params = []

    for (
        process,
        status,
        last_success,
        retry_count,
        failure_message,
        run_id,
        run_info
    ) in results:

        row_clauses.append(row_clause)
        params.extend([process, status, last_success, retry_count])

    merge_sql = f"""
    MERGE orch.processexecutionstatus AS target
    USING (
        VALUES
        {', '.join(row_clauses)}
    ) AS source (
        Process,
        Status,
        LastSuccess,
        RetryCount
    )
    ON target.Process = source.Process

    WHEN MATCHED THEN
        UPDATE SET
            Status = source.Status,
            LastSuccess = CASE
                WHEN source.Status = 'Completed'
                THEN source.LastSuccess
                ELSE target.LastSuccess
            END,
            RetryCount = source.RetryCount,
            LastCheckedTime = CURRENT_TIMESTAMP;
    """

    execute_sql(merge_sql, params)

    for (
        proc,
        status,
        last_success,
        retry_count,
        failure_message,
        run_id,
        run_info
    ) in results:

        if status == "Completed":
            log_event(
                proc,
                "Success",
                f"Run succeeded! {run_info}",
                run_id
            )
        elif status == "Failed":
            log_event(
                proc,
                "Failed",
                failure_message,
                run_id
            )
        else:
            log_event(
                proc,
                "Info",
                "Still running",
                run_id
            )

Takes a process name, last_modified, and retry_count. Updates the status table for the process to "Running". Triggers the pipeline for the given process.

Runs on the thread pool - all SQL goes through execute_sql, which serializes on
sql_lock, so no additional locking is needed here.

In [ ]:
FAIL_PROCESS_SQL = """
UPDATE orch.processexecutionstatus
SET
    Status = 'Failed',
    RetryCount = ?
WHERE Process = ?
"""


def run_process(process, isGold, lookbackRequired, last_modified, retry_count):
    # A process with no row in processexecutionstatus yet arrives here with
    # retry_count None; coalesce so the failure paths below can increment it.
    retry_count = retry_count or 0

    try:
        proc_info = dependency_map.get(process)

        if not proc_info:
            raise ValueError(f"No dependency info found for {process}")

        workspace_prefix = proc_info["WorkspacePrefix"]
        is_stored_proc = proc_info["isStoredPro"]

        print(f"Triggering pipeline: {process}", flush=True)

        success, run_id = trigger_pipeline(
            process,
            isGold,
            lookbackRequired,
            workspace_prefix,
            is_stored_proc,
            last_modified
        )

        if success:
            execute_sql(
                """
                MERGE orch.processexecutionstatus AS target
                USING (
                    SELECT
                        CAST(? AS VARCHAR(4000)) AS Process,
                        CAST(? AS VARCHAR(4000)) AS RunInstanceId
                ) AS source
                ON target.Process = source.Process

                WHEN MATCHED THEN
                    UPDATE SET
                        Status = 'Running',
                        LastRunTime = CURRENT_TIMESTAMP,
                        RunInstanceId = source.RunInstanceId

                WHEN NOT MATCHED THEN
                    INSERT (
                        Process,
                        Status,
                        LastRunTime,
                        RunInstanceId
                    )
                    VALUES (
                        source.Process,
                        'Running',
                        CURRENT_TIMESTAMP,
                        source.RunInstanceId
                    );
                """,
                (process, run_id)
            )

        else:
            execute_sql(FAIL_PROCESS_SQL, (retry_count + 1, process))

    except Exception as e:
        execute_sql(FAIL_PROCESS_SQL, (retry_count + 1, process))

        print(f"Error running {process}: {e}", flush=True)

        log_event(
            process,
            "Failed",
            f"Failed to trigger process: {e}"
        )

Minimal cron evaluator for `RunTimes`. Supports the standard 5 fields (minute hour day-of-month month day-of-week) with `*`, comma lists, ranges (a-b) and steps (*/n or a-b/n). `latest_cron_run` returns the most recent minute <= now that matches, replacing the old bare comma-separated-hour parsing.

In [ ]:
def _cron_field_matches(value, field, min_val, max_val):
    field = field.strip()
    if field == "*":
        return True

    for part in field.split(","):
        part = part.strip()
        if not part:
            continue

        step = 1
        if "/" in part:
            part, step_str = part.split("/", 1)
            step = int(step_str)

        if part == "*":
            lo, hi = min_val, max_val
        elif "-" in part:
            lo, hi = (int(x) for x in part.split("-", 1))
        else:
            lo = hi = int(part)

        if lo <= value <= hi and (value - lo) % step == 0:
            return True

    return False


def cron_matches(cron_expr, dt):
    fields = cron_expr.split()
    if len(fields) != 5:
        raise ValueError(f"Expected 5 cron fields (minute hour dom month dow), got {cron_expr!r}")

    minute, hour, dom, month, dow = fields

    if not _cron_field_matches(dt.minute, minute, 0, 59):
        return False
    if not _cron_field_matches(dt.hour, hour, 0, 23):
        return False
    if not _cron_field_matches(dt.month, month, 1, 12):
        return False

    dom_restricted = dom.strip() != "*"
    dow_restricted = dow.strip() != "*"
    dom_match = _cron_field_matches(dt.day, dom, 1, 31)
    # Cron day-of-week: 0 and 7 both mean Sunday; Python's weekday() is Mon=0..Sun=6.
    dow_value = (dt.weekday() + 1) % 7
    dow_match = _cron_field_matches(dow_value, dow, 0, 7) or (
        dow_value == 0 and _cron_field_matches(7, dow, 0, 7)
    )

    # POSIX semantics: when both day-of-month and day-of-week are restricted,
    # a match on EITHER is enough.
    if dom_restricted and dow_restricted:
        return dom_match or dow_match
    return dom_match and dow_match


def latest_cron_run(cron_expr, now):
    """Most recent minute today (<= now) matching `cron_expr`, or None if none yet today.

    Bounded to today only, matching the previous hour-list behaviour: a
    process becomes eligible again once today's scheduled time has passed,
    not based on some older occurrence from a prior day.
    """
    candidate = now.replace(second=0, microsecond=0)
    start_of_day = now.replace(hour=0, minute=0, second=0, microsecond=0)

    while candidate >= start_of_day:
        if cron_matches(cron_expr, candidate):
            return candidate
        candidate -= timedelta(minutes=1)

    return None


_latest_cron_run_cache = {}


def latest_cron_run_cached(cron_expr, now):
    """Memoized `latest_cron_run`. `now` is frozen for the whole run.

    dependencygraph is an edge list, so many rows repeat a handful of distinct
    cron strings. Caching collapses the minute-by-minute scan from
    O(rows * 1440) to O(distinct expressions * 1440).
    """
    if cron_expr not in _latest_cron_run_cache:
        _latest_cron_run_cache[cron_expr] = latest_cron_run(cron_expr, now)

    return _latest_cron_run_cache[cron_expr]

In [ ]:
try:
    # Naive UTC (see SCHEDULE_TIMEZONE): same frame as LastSuccess and as the
    # warehouse's CURRENT_TIMESTAMP, so the eligibility comparison below and the
    # cron evaluation agree with each other.
    now = datetime.now(SCHEDULE_TIMEZONE).replace(tzinfo=None)

    # Load dependency graph
    dependency_rows = fetch_as_dicts("""
    SELECT *
    FROM orch.dependencygraph
    """)

    # Filter active processes
    dependency_rows = [
        row for row in dependency_rows
        if row["isActive"]
    ]

    # Environment filtering
    if current_env.upper() == "UAT":
        dependency_rows = [
            row for row in dependency_rows
            if row["readyForUAT"]
        ]

    elif current_env.upper() == "PRD":
        dependency_rows = [
            row for row in dependency_rows
            if row["readyForProd"]
        ]

    # Calculate LatestRunTime
    for row in dependency_rows:

        run_times = row.get("RunTimes")
        latest_run = None

        if run_times:
            try:
                latest_run = latest_cron_run_cached(run_times.strip(), now)
            except Exception as e:
                print(
                    f"Invalid RunTimes cron expression for {row.get('Process')}: "
                    f"{run_times!r} ({e})",
                    flush=True
                )

        row["LatestRunTime"] = latest_run

    # Remove circular dependencies
    dependency_rows, removed = break_cycles(dependency_rows)

    if removed:
        print("Circular dependencies detected and removed:")

        for dep, proc in removed:
            print(f"Removed edge: {dep} → {proc}")
            log_event(proc, "Removed", "Circular dependency")

    # Build lookup structures
    dependency_map = {
        row["Process"]: {
            "WorkspacePrefix": row["WorkspacePrefix"],
            "isStoredPro": row["isStoredPro"],
            "isGold": row["isGold"]
        }
        for row in dependency_rows
    }

    gold_map = {
        row["Process"]: row["isGold"]
        for row in dependency_rows
    }

    lookback_required_map = {
        row["Process"]: row["lookbackRequired"]
        for row in dependency_rows
    }

    latest_run_map = {
        row["Process"]: row["LatestRunTime"]
        for row in dependency_rows
    }

    dependencies_by_process = {}

    for row in dependency_rows:

        dependency = row.get("Dependency")

        if dependency:
            dependencies_by_process.setdefault(
                row["Process"],
                []
            ).append(dependency)

    # Refresh currently running process statuses
    poll_running_processes()

    # Load process status table
    status_rows = fetch_as_dicts("""
    SELECT
        Process,
        Status,
        LastSuccess,
        RetryCount
    FROM orch.processexecutionstatus
    """)

    status_map = {
        row["Process"]: row
        for row in status_rows
    }

    last_modified_map = {
        row["Process"]: row["LastSuccess"]
        for row in status_rows
    }

    retry_count_map = {
        row["Process"]: (row["RetryCount"] or 0)
        for row in status_rows
    }

    completed_map = {
        row["Process"]: row["LastSuccess"]
        for row in status_rows
        if row["Status"] == "Completed"
    }

    # Determine eligible processes
    eligible_processes = set()

    for row in dependency_rows:

        process = row["Process"]

        status_row = status_map.get(process)

        status = status_row["Status"] if status_row else None
        last_success = status_row["LastSuccess"] if status_row else None

        latest_run = row["LatestRunTime"]

        not_running = (
            status is None
            or status != "Running"
        )

        needs_run = (
            last_success is None
            or (
                latest_run is not None
                and last_success < latest_run
            )
        )

        if not_running and needs_run:
            eligible_processes.add(process)

    eligible_processes = list(eligible_processes)

    # Apply retry limits
    retry_blocked = apply_retry_limits(
        eligible_processes,
        retry_count_map
    )

    eligible_processes = [
        proc
        for proc in eligible_processes
        if proc not in retry_blocked
    ]

    # Dependency validation
    ready_to_run = []

    for proc in eligible_processes:

        deps = dependencies_by_process.get(proc, [])

        if not deps:
            ready_to_run.append(proc)
            continue

        all_satisfied = True

        for dep_name in deps:

            dep_latest = latest_run_map.get(dep_name)
            dep_completed = completed_map.get(dep_name)

            if dep_completed is None:
                all_satisfied = False
                break

            if dep_latest is None:
                continue

            if dep_completed < dep_latest:
                all_satisfied = False
                break

        if all_satisfied:
            ready_to_run.append(proc)

    # Reset status for dependency-blocked processes
    blocked_by_deps = (
        set(eligible_processes)
        - set(ready_to_run)
    )

    if blocked_by_deps:

        print(
            f"Blocked by dependencies (status set to NULL): {blocked_by_deps}",
            flush=True
        )

        blocked = list(blocked_by_deps)
        placeholders = ", ".join("?" for _ in blocked)

        execute_sql(
            f"""
            UPDATE orch.processexecutionstatus
            SET Status = NULL
            WHERE Process IN ({placeholders})
            """,
            blocked
        )

    print(f"Ready to run: {ready_to_run}", flush=True)

    # Execute processes in parallel
    with ThreadPoolExecutor() as executor:

        futures = {
            executor.submit(
                run_process,
                process,
                gold_map.get(process),
                lookback_required_map.get(process),
                last_modified_map.get(process),
                retry_count_map.get(process)
            ): process
            for process in ready_to_run
        }

        for future in as_completed(futures):

            process = futures[future]

            try:
                future.result()

            except Exception as e:
                print(
                    f"[{process}] failed with error: {e}",
                    flush=True
                )

finally:

    execute_sql("""
    UPDATE orch.orchestrator_lock
    SET AcquiredAt = CAST('1900-01-01 00:00:00' AS DATETIME2)
    WHERE LockName = 'orchestrator'
    """)

    print("Lock released.")